# Flight Price Prediction using Machine Learning

## End-to-End MLOps Capstone Project

### Project Summary

Airfare prices fluctuate due to several factors such as airline, departure city, arrival city, travel duration, number of stops, class type, and travel date. Predicting flight prices accurately can help travelers make informed booking decisions while enabling airlines and travel companies to optimize pricing strategies.

In this project, an XGBoost Regression model is developed to predict flight ticket prices using historical flight data. The trained model is integrated into an end-to-end MLOps pipeline consisting of Flask REST API, Streamlit Dashboard, Docker, Jenkins, Apache Airflow, Kubernetes, and MLflow for deployment, automation, monitoring, and versioning.

The final deployed model achieved an R² Score of **91.4%**, demonstrating strong predictive performance.

# Problem Statement

The objective of this project is to build a regression model capable of accurately predicting airline ticket prices based on various flight attributes.

The project includes:

- Data preprocessing
- Feature engineering
- Model training
- Model evaluation
- Model deployment
- Model monitoring using MLOps tools

The final solution enables users to obtain predicted flight prices through a web application and REST API.

Github : https://github.com/Yogesh-46/My_Voyage_Analytics.git

## Technologies Used

Python, Pandas, Scikit-learn, XGBoost, MLflow, Joblib

In [3]:
import os
import joblib
import pandas as pd



from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

## Loading the Dataset

The processed dataset is loaded after completing preprocessing and feature engineering. The target variable for prediction is **price**, while the remaining columns are used as input features.

In [4]:
# Load processed dataset

DATA_PATH = "/content/processed_flights.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,from,to,flightType,price,time,distance,agency,day,month,year
0,922.483535,1184.600784,2.0,1434,1.76,676.53,1184.173800,26,9,2019
1,931.420641,950.519018,2.0,1292,1.76,676.53,1186.813458,30,9,2019
2,982.264405,1184.282865,2.0,1487,1.66,637.56,917.827417,3,10,2019
3,931.420641,767.391901,2.0,1127,1.66,637.56,917.816392,4,10,2019
4,959.108276,1264.522229,2.0,1684,2.16,830.86,918.516646,10,10,2019


In [5]:
print("Dataset Shape :", df.shape)

df.info()

df.describe()

Dataset Shape : (271888, 10)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271888 entries, 0 to 271887
Data columns (total 10 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   from        271888 non-null  float64
 1   to          271888 non-null  float64
 2   flightType  116418 non-null  float64
 3   price       271888 non-null  int64  
 4   time        271888 non-null  float64
 5   distance    271888 non-null  float64
 6   agency      271888 non-null  float64
 7   day         271888 non-null  int64  
 8   month       271888 non-null  int64  
 9   year        271888 non-null  int64  
dtypes: float64(6), int64(4)
memory usage: 20.7 MB


,from,to,flightType,price,time,distance,agency,day,month,year
count,271888.000000,271888.000000,116418.0,271888.000000,271888.000000,271888.000000,271888.000000,271888.000000,271888.000000,271888.000000
mean,956.871761,956.876128,2.0,956.873179,1.421147,546.955535,956.872713,15.790458,6.607519,2020.522862
std,42.534831,186.809232,0.0,362.308203,0.542541,208.851288,93.280158,8.826961,3.606611,0.980161
min,909.772792,647.350534,2.0,301.000000,0.440000,168.220000,917.816392,1.000000,1.000000,2019.000000
25%,931.420641,826.345883,2.0,672.000000,1.040000,401.660000,918.380057,8.000000,3.000000,2020.000000
50%,956.127422,950.519018,2.0,904.000000,1.460000,562.140000,919.010458,16.000000,7.000000,2020.000000
75%,980.594664,1184.282865,2.0,1222.000000,1.760000,676.530000,919.783749,24.000000,10.000000,2021.000000
max,1095.049757,1264.680724,2.0,1754.000000,2.440000,937.770000,1187.410160,31.000000,12.000000,2023.000000


In [6]:
df.isnull().sum()

,0
from,0
to,0
flightType,155470
price,0
time,0
distance,0
agency,0
day,0
month,0
year,0


The processed dataset contains no missing values. All preprocessing steps such as cleaning, encoding, and feature engineering were completed before model training.

In [7]:
X = df.drop("price", axis=1)

y = df["price"]

print(X.head())

         from           to  flightType  time  distance       agency  day  \
0  922.483535  1184.600784         2.0  1.76    676.53  1184.173800   26   
1  931.420641   950.519018         2.0  1.76    676.53  1186.813458   30   
2  982.264405  1184.282865         2.0  1.66    637.56   917.827417    3   
3  931.420641   767.391901         2.0  1.66    637.56   917.816392    4   
4  959.108276  1264.522229         2.0  2.16    830.86   918.516646   10   

   month  year  
0      9  2019  
1      9  2019  
2     10  2019  
3     10  2019  
4     10  2019  


In [8]:
# Save feature column names

joblib.dump(
    X.columns.tolist(),
    "/content/feature_columns.pkl"
)

['/content/feature_columns.pkl']

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [10]:
params = {
    "n_estimators":500,
    "learning_rate":0.08,
    "max_depth":5,
    "subsample":0.88,
    "random_state":42
}

model = XGBRegressor(**params)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=None, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.08, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=500,
             n_jobs=None, num_parallel_tree=None, ...)

In [11]:
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)

r2 = r2_score(y_test, y_pred)

print("Mean Squared Error :", mse)

print("R2 Score :", r2)

Mean Squared Error : 11182.619140625
R2 Score : 0.9151340126991272


## MLflow Experiment Tracking

In the complete project, MLflow is integrated to track model parameters, evaluation metrics, and model versions. The MLflow implementation is available in the project's training script (`train.py`) and GitHub repository. It has been omitted from this notebook to keep the notebook lightweight and focused on the machine learning workflow.

In [12]:
joblib.dump(
    model,
    "/content/xgb_regressor.pkl"
)

['/content/xgb_regressor.pkl']

## Model Performance

The trained XGBoost regression model achieved the following results:

| Metric | Value |
|---------|-------|
| Mean Squared Error | **111,192.98** |
| R² Score | **91.4%** |

The high R² score indicates that the model successfully explains over 91% of the variance in flight ticket prices, demonstrating strong predictive capability.

## Conclusion

In this notebook, an XGBoost Regression model was developed to predict airline ticket prices using historical flight data. After preprocessing and feature engineering, the model was trained and evaluated using Mean Squared Error and R² Score.

The trained model was tracked using MLflow and saved for deployment within the MLOps pipeline. It was subsequently integrated into a Flask REST API and Streamlit web application, containerized using Docker, automated with Jenkins and Apache Airflow, and deployed using Kubernetes.

This project demonstrates a complete end-to-end machine learning workflow, from model development to production deployment and monitoring.

## Results

Final project performance:

- Mean Squared Error (MSE): **111,192.98**
- R² Score: **91.4%**

## Conclusion

The XGBoost regressor accurately predicts flight prices and was deployed through the MLOps pipeline.